# Phase 2: Data Cleaning & Integrity
**Project:** Customer Churn Intelligence  
**Objective:** Convert the raw IBM Telco Customer Churn dataset into a clean, leakage-free, reliable analytical dataset.

---
### Guiding Principles for Data Cleaning
1. **Preserve Raw Data:** The raw CSV file in `data/raw/` remains untouched.
2. **Explain Every Transformation:** Every cleaning step must have an explicit business or technical rationale.
3. **Prevent Data Leakage:** No statistical transformations dependent on future splits (e.g., standard scaling, train/test splitting, target-based imputation) are conducted here.


## Step 1 — Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load the raw dataset
raw_data_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(raw_data_path)
print(f"Loaded raw data with shape: {df.shape}")
df.head(3)


Loaded raw data with shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## Step 2.1 — Remove Irrelevant ID (`customerID`)
**Rationale:**  
`customerID` is an arbitrary unique identifier generated by the billing system. It has 7,043 distinct alphanumeric values with zero inherent predictive value. Keeping it would risk overfitting or high cardinality bloat.

We retain a copy or record of customer IDs if needed for audit, but drop it from analytical modeling.

In [2]:
print(f"Total rows: {len(df)}, Unique customer IDs: {df['customerID'].nunique()}")

# Drop customerID
df = df.drop(columns=["customerID"])
print(f"Shape after dropping customerID: {df.shape}")


Total rows: 7043, Unique customer IDs: 7043
Shape after dropping customerID: (7043, 20)


## Step 2.2 — Fix `TotalCharges` Data Type & Investigate Missing Values
Initially, `TotalCharges` is stored as `object` (string) rather than floating point numbers.  
Let us inspect why it was parsed as string and convert it with `errors='coerce'` to expose hidden missing or malformed values.

In [3]:
print(f"Initial dtype of TotalCharges: {df['TotalCharges'].dtype}")

# Convert TotalCharges to numeric, coercing unparseable strings (e.g. whitespace) to NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(f"New dtype of TotalCharges: {df['TotalCharges'].dtype}")
print(f"Number of NaN values generated: {df['TotalCharges'].isna().sum()}")


Initial dtype of TotalCharges: str
New dtype of TotalCharges: float64
Number of NaN values generated: 11


### Deep Investigation: Why are these 11 rows missing?
Rather than blindly filling `TotalCharges` with 0 or the mean/median, let us inspect the entire record for these 11 customers.

In [4]:
# Inspect the rows with NaN in TotalCharges
missing_tc = df[df["TotalCharges"].isna()]
print(f"Number of affected rows: {len(missing_tc)}")
missing_tc[["tenure", "MonthlyCharges", "TotalCharges", "Contract", "PaymentMethod", "Churn"]]


Number of affected rows: 11


,tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,Churn
488,0,52.55,NaN,Two year,Bank transfer (automatic),No
753,0,20.25,NaN,Two year,Mailed check,No
936,0,80.85,NaN,Two year,Mailed check,No
1082,0,25.75,NaN,Two year,Mailed check,No
1340,0,56.05,NaN,Two year,Credit card (automatic),No
3331,0,19.85,NaN,Two year,Mailed check,No
3826,0,25.35,NaN,Two year,Mailed check,No
4380,0,20.00,NaN,Two year,Mailed check,No
5218,0,19.70,NaN,One year,Mailed check,No
6670,0,73.35,NaN,Two year,Mailed check,No


In [5]:
# Check tenure distribution for these 11 records
print("Tenure values for missing TotalCharges rows:")
print(missing_tc["tenure"].value_counts())


Tenure values for missing TotalCharges rows:
tenure
0    11
Name: count, dtype: int64


**Critical Domain Finding:**  
Every single customer with missing `TotalCharges` has `tenure == 0`.  
- These are brand-new customers who signed contracts (1-year or 2-year) during the current billing cycle.
- Because they have been with the company for less than one month (`tenure == 0`), they have **not yet received or paid their first monthly invoice**.
- Therefore, their cumulative billed charges to date are legitimately **$0.00**.
- **Action:** Impute `TotalCharges = 0.0` for these 11 records. This aligns with domain logic and avoids dropping valid customer observations.

In [6]:
# Impute TotalCharges = 0.0 for tenure == 0
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

print(f"TotalCharges missing count after imputation: {df['TotalCharges'].isna().sum()}")
print("Verification for tenure == 0 rows:")
df[df["tenure"] == 0][["tenure", "MonthlyCharges", "TotalCharges", "Churn"]]


TotalCharges missing count after imputation: 0
Verification for tenure == 0 rows:


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,0.0,No
753,0,20.25,0.0,No
936,0,80.85,0.0,No
1082,0,25.75,0.0,No
1340,0,56.05,0.0,No
3331,0,19.85,0.0,No
3826,0,25.35,0.0,No
4380,0,20.00,0.0,No
5218,0,19.70,0.0,No
6670,0,73.35,0.0,No


## Step 2.3 — Check Missing Values Across All Columns
We verify that no other columns contain missing values.

In [7]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': (missing.values / len(df) * 100).round(3)
})
missing_summary[missing_summary['Missing Count'] > 0]


,Column,Missing Count,Missing %


### Missing Value Documentation Table
| Column | Missing Count | Missing % | Action Taken | Rationale |
| :--- | :--- | :--- | :--- | :--- |
| `TotalCharges` | 11 | 0.156% | Imputed with `0.0` | All 11 records have `tenure == 0` (new signups prior to first billing cycle). Total billed to date is legitimately $0. |
| All other columns | 0 | 0.0% | None | Complete data across all demographic, service, and account fields. |


## Step 2.4 — Check Duplicate Records
We inspect whether exact duplicate rows exist after removing `customerID`.

In [8]:
dup_count = df.duplicated().sum()
print(f"Number of exact duplicate rows: {dup_count}")

if dup_count > 0:
    duplicates = df[df.duplicated(keep=False)].sort_values(by=list(df.columns[:5]))
    print(f"Sample duplicates:\n{duplicates.head(4)}")
    # In telco data, identical service configurations across distinct real customers can naturally occur once customerID is removed.
    # We inspect if they are genuine duplicate submissions or identical customer subscription tiers.


Number of exact duplicate rows: 22
Sample duplicates:
      gender  SeniorCitizen Partner Dependents  tenure PhoneService MultipleLines InternetService       OnlineSecurity         OnlineBackup     DeviceProtection          TechSupport          StreamingTV      StreamingMovies        Contract PaperlessBilling PaymentMethod  MonthlyCharges  TotalCharges Churn
542   Female              0      No         No       1          Yes            No              No  No internet service  No internet service  No internet service  No internet service  No internet service  No internet service  Month-to-month               No  Mailed check           19.55         19.55    No
1491  Female              0      No         No       1          Yes            No              No  No internet service  No internet service  No internet service  No internet service  No internet service  No internet service  Month-to-month               No  Mailed check           19.55         19.55    No
3499  Female             

**Investigation on Duplicates:**  
There are 22 duplicate rows out of 7,043 once `customerID` is removed. In subscription services, identical demographic and plan profiles (e.g. month-to-month single male without add-ons) can exist among different human customers. Dropping or keeping them: since `customerID` was originally 100% unique (7,043 distinct customers), each row represents an independent customer account. We drop exact duplicates only if confirmed as duplicate data entry errors; however, keeping distinct customer contracts is common practice. To ensure strict analytical purity, let's keep all 7,043 distinct customer accounts or document their distribution.

## Step 2.5 — Check Invalid Values & Range Validity
Inspect numerical columns (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`) for:
- Negative values
- Impossible values
- Unexpected zeros
- Extreme anomalies

In [9]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
num_stats = df[num_cols].describe().T
num_stats["min_valid"] = num_stats["min"] >= 0
num_stats["max_valid"] = num_stats["max"] > 0
num_stats


,count,mean,std,min,25%,50%,75%,max,min_valid,max_valid
tenure,7043.0,32.371149,24.559481,0.00,9.00,29.00,55.00,72.00,True,True
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.50,70.35,89.85,118.75,True,True
TotalCharges,7043.0,2279.734304,2266.794470,0.00,398.55,1394.55,3786.60,8684.80,True,True


In [10]:
# Sanity checks
assert (df["tenure"] >= 0).all(), "Found negative tenure!"
assert (df["tenure"] <= 72).all(), "Found tenure > 72 months (6 years)!"
assert (df["MonthlyCharges"] > 0).all(), "Found non-positive MonthlyCharges!"
assert (df["TotalCharges"] >= 0).all(), "Found negative TotalCharges!"
assert set(df["SeniorCitizen"].unique()).issubset({0, 1}), "SeniorCitizen contains values other than 0 and 1!"

print("All numerical range validations passed successfully!")


All numerical range validations passed successfully!


## Step 2.6 — Check Categorical Consistency
We examine the unique values across every categorical column to verify:
- No trailing/leading whitespaces (e.g. `' Yes '` vs `'Yes'`)
- No inconsistent casing (e.g. `'yes'` vs `'Yes'`)
- Clear, standardized categories

In [11]:
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_columns:
    unique_vals = df[col].unique().tolist()
    print(f"Column: {col:<18} | Distinct: {len(unique_vals)} | Values: {unique_vals}")


Column: gender             | Distinct: 2 | Values: ['Female', 'Male']
Column: Partner            | Distinct: 2 | Values: ['Yes', 'No']
Column: Dependents         | Distinct: 2 | Values: ['No', 'Yes']
Column: PhoneService       | Distinct: 2 | Values: ['No', 'Yes']
Column: MultipleLines      | Distinct: 3 | Values: ['No phone service', 'No', 'Yes']
Column: InternetService    | Distinct: 3 | Values: ['DSL', 'Fiber optic', 'No']
Column: OnlineSecurity     | Distinct: 3 | Values: ['No', 'Yes', 'No internet service']
Column: OnlineBackup       | Distinct: 3 | Values: ['Yes', 'No', 'No internet service']
Column: DeviceProtection   | Distinct: 3 | Values: ['No', 'Yes', 'No internet service']
Column: TechSupport        | Distinct: 3 | Values: ['No', 'Yes', 'No internet service']
Column: StreamingTV        | Distinct: 3 | Values: ['No', 'Yes', 'No internet service']
Column: StreamingMovies    | Distinct: 3 | Values: ['No', 'Yes', 'No internet service']
Column: Contract           | Distinct: 3 |

C:\Users\USER\AppData\Local\Temp\ipykernel_31184\1396127276.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()


**Categorical Consistency Finding:**  
All categories are consistent, clean, and well-structured. No trailing whitespaces or irregular casing was detected across any categorical attribute.

## Step 2.7 — Target Encoding (`Churn`)
We map the target variable:
- `'Yes'` -> `1` (Customer churned)
- `'No'` -> `0` (Customer retained)

In [12]:
# Map Churn to binary integer
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Mapped Churn value counts:")
print(df["Churn"].value_counts())
print("\nMapped Churn percentage:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

assert df["Churn"].isin([0, 1]).all(), "Target contains values outside [0, 1]!"


Mapped Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64

Mapped Churn percentage:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


## Step 2.8 — Explicit Data Leakage & Modeling Safeguards
In accordance with production ML best practices, we strictly **do not** perform the following during Phase 2:
- ❌ **SMOTE / Oversampling:** Oversampling before train/test splitting causes synthetic test points to leak into training distributions.
- ❌ **Train / Test Split:** Cleaning applies to data formatting; train/test splitting is part of the modeling pipeline.
- ❌ **StandardScaler / MinMax Scaling:** Fitting scalers across the full dataset causes distribution leakage.
- ❌ **Model Training:** No predictive models are trained yet.
- ❌ **Target-based Feature Selection:** Correlation or mutual information with target must not be used to discard features before cross-validation.
- ❌ **XGBoost / Random Forest:** Reserved strictly for Phase 5.


## Step 2.9 — Save Cleaned Analytical Dataset
We export the verified, clean dataset to `../data/processed/telco_churn_clean.csv` to serve as the single source of truth for Phase 3 (Business EDA) and Phase 4 (Feature Engineering).

In [13]:
output_clean_path = "../data/processed/telco_churn_clean.csv"
df.to_csv(output_clean_path, index=False)
print(f"Cleaned dataset successfully saved to: {output_clean_path}")
print(f"Final shape: {df.shape}")
df.info()


Cleaned dataset successfully saved to: ../data/processed/telco_churn_clean.csv
Final shape: (7043, 20)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 1